# OSFT Continual Learning - Lightweight Functional Test NB

The following is an adaptation of our original [continual learning demo](https://github.com/Red-Hat-AI-Innovation-Team/training_hub/blob/main/examples/notebooks/osft_continual_learning.ipynb), and has been reduced to ensure all cells are runnable ootb and training is completeable on 2xL40/L40S (2x48GB) GPUs within 15-20 minutes. Note that this can fit on some smaller configurations, and can run much faster on larger/newer configurations. Please adjust `nproc_per_node` to the number of GPUs (default is 2), and reduce `max_tokens_per_gpu` if necessary given total memory available.

## Continual Learning Demo

Fine-tuning language models is hard—you need good data, lots of resources, and even small changes can cause problems. This makes it tough to add new abilities to a model. This problem is called **continual learning** and is what our new training technique, orthogonal subspace fine-tuning (OSFT), solves.

This notebook presents a hands-on example where we enhance `Qwen/Qwen2.5-1.5B-Instruct` by teaching it to only produce JSON output when requested.

By the end of this notebook, you will learn:
- ✅ How Qwen can be fine-tuned without destroying its existing capabilities
- ✅ How to enhance your own LLMs with OSFT
- ✅ Best practices when fine-tuning models
- ❌ OSFT does NOT kill your existing model when trained on new data


**Step 1: Environment Setup**
First, we will install our necessary training dependencies (like `training-hub`) and configure our environment logging. training-hub can tak eosme time to finish 10 min,

In [ ]:
!pip install training-hub[cuda] --no-build-isolation

In [ ]:
!pip install lm-eval[api] langdetect immutabledict

In [ ]:
# Import training_hub for OSFT training
from training_hub import osft

# Standard library imports
import os
import time
import logging
import sys
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO


In [ ]:
# Configure logging to show only essential information
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)

# Suppress verbose logging from transformers and other libraries
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("datasets").setLevel(logging.WARNING)
logging.getLogger("torch").setLevel(logging.WARNING)

print("✅ Logging configured for notebook environment")


In [ ]:
import glob


def find_most_recent_checkpoint(output_dir):
    """
    Find the most recent checkpoint in the training output directory.
    
    Args:
        output_dir (str): Training output directory containing hf_format/ subdirectory
        
    Returns:
        str: Path to the most recent checkpoint
        
    Raises:
        ValueError: If no checkpoints are found
    """
    # Get all checkpoint directories under hf_format
    checkpoint_pattern = os.path.join(output_dir, "hf_format", "samples_*.0")
    checkpoint_dirs = glob.glob(checkpoint_pattern)
    
    if not checkpoint_dirs:
        raise ValueError(f"No checkpoints found in {os.path.join(output_dir, 'hf_format')}")
    
    # Find the most recently created checkpoint
    most_recent_checkpoint = max(checkpoint_dirs, key=os.path.getctime)
    
    return most_recent_checkpoint


print("✅ Checkpoint utility functions defined")



### 📂 Step 2: Dynamic Data Ingestion & Formatting
Below, we have built a **Universal Data Loader**. 

By changing the `ACTIVE_SCENARIO` variable, we will map the raw data from Hugging Face, extract the relevant columns (whether it's tabular data or nested conversation logs), and format it into the exact JSONL structure required for model training. 

*Try changing the scenario to instantly pivot the demo from Cybersecurity, to Medical, to First Aid*

In [ ]:
import os
import json
from datasets import load_dataset

# ==============================================================================
# 🎯 DEMO SCENARIO SELECTOR
# Options: "medical_summary", "cyber_analysis", "first_aid"
# ==============================================================================
ACTIVE_SCENARIO = "cyber_analysis"  

SCENARIOS = {
    "medical_summary": {
        "dataset_name": "ccdv/pubmed-summarization",
        "split": "train",
        "train_size": 1000,
        "eval_start": 1600,
        "input_col": "article",     
        "target_col": "abstract",   
        "system_prompt": "You are a senior medical researcher. Summarize the following clinical study for a peer review.",
        "user_prefix": "Please summarize this article:\n\n"
    },
    "cyber_analysis": {
        "dataset_name": "Tiamz/cybersecurity-instruction-dataset",
        "split": "train",
        "train_size": 1000,
        "eval_start": 1600,
        "input_col": "instruction",  
        "target_col": "answer",      
        "system_prompt": "You are a Senior Cyber Intrusion Analyst. Provide precise, actionable security incident analysis, mitigation steps, and threat intelligence.",
        "user_prefix": "Security Alert/Inquiry: " 
    },
    "first_aid": {
        "dataset_name": "i-am-mushfiq/FirstAidQA",
        "split": "train",
        "train_size": 1000,
        "eval_start": 1600,
        "input_col": "question", 
        "target_col": "answer",  
        "system_prompt": "You are an emergency first-aid responder. Provide clear, step-by-step, and safe instructions.",
        "user_prefix": "" 
    }
}

CONF = SCENARIOS[ACTIVE_SCENARIO]
print(f"✅ Demo configured for: {ACTIVE_SCENARIO.upper()}")
print(f"📂 Loading {CONF['dataset_name']} (Streaming)...")

raw_dataset = load_dataset(CONF['dataset_name'], split=CONF['split'], streaming=True)
formatted_data = []
print("🔄 Formatting for Training Hub (OpenAI Messages Format)...")

for i, row in enumerate(raw_dataset.take(CONF["train_size"])):
    user_input = f"{CONF['user_prefix']}{row[CONF['input_col']]}"
    expected_output = row[CONF['target_col']]
    
    formatted_data.append({
        "messages": [
            {"role": "system", "content": CONF["system_prompt"]},
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": expected_output},
        ]
    })

# Dynamic paths to keep data completely isolated per industry
DATASET_PATH = f"demo-data/{ACTIVE_SCENARIO}/formatted_training_data.jsonl"
CHECKPOINTS_PATH = f"checkpoints-{ACTIVE_SCENARIO}"
os.makedirs(f"demo-data/{ACTIVE_SCENARIO}", exist_ok=True)

with open(DATASET_PATH, "w", encoding="utf-8") as f:
    for example in formatted_data:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")
        
print(f"✅ Saved {len(formatted_data)} records to {DATASET_PATH}")

### ⚙️ Step 3: Hardware-Optimized Configuration
In the next cell, we expose the training hyperparameters. 

We have optimized these settings (such as lowering `MAX_TOKENS_PER_GPU` and `BATCH_SIZE`) to ensure safe, efficient memory management for our specific hardware (L40S GPU). Change these settings to match your hardware.

In [ ]:
################################################################################
# 🤖 Model + Data Paths                                                        #
################################################################################
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# DATASET_PATH = "table-gpt-data/train/train_All_5000.jsonl"
# CHECKPOINTS_PATH = "checkpoints-logs-dir"
DATA_OUTPUT_PATH = "/dev/shm"  # for quicker multi-process loading of datasets


################################################################################
# 🏋️‍♀️ Training Hyperparameters                                                  #
################################################################################
# Important for OSFT
UNFREEZE_RANK_RATIO = 0.25 

# Standard parameters
BATCH_SIZE = 128
LEARNING_RATE = 5e-6
NUM_EPOCHS=3
LR_SCHEDULER="cosine"
WARMUP_STEPS=0
SEED=42


################################################################################
# 🏎️ Performance Hyperparameters                                               #
################################################################################
USE_LIGER = True
MAX_TOKENS_PER_GPU=64000
MAX_SEQ_LEN=8192

################################################################################
# 💾 Checkpointing Settings                                                    #
################################################################################
# Here we only want to save the very last checkpoint
SAVE_FINAL_CHECKPOINT = True
CHECKPOINT_AT_EPOCH = False 

################################################################################
# 🔥 TORCHRUN SETTINGS                                                         #
################################################################################
NUM_GPUS=1
NUM_NODES=1
NODE_RANK=0
RDZV_ID=23
RDZV_ENDPOINT='localhost:1738'


print("⚙️  Training Hyperparameters")
print("=" * 50)
print(f"Base Model: {BASE_MODEL}")
print(f"Dataset Path: {DATASET_PATH}")
print(f"Checkpoints Path: {CHECKPOINTS_PATH}")
print(f"Data Output Path: {DATA_OUTPUT_PATH}")
print()
print(f"Unfreeze Rank Ratio: {UNFREEZE_RANK_RATIO}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Number of Epochs: {NUM_EPOCHS}")
print(f"LR Scheduler: {LR_SCHEDULER}")
print(f"Warmup Steps: {WARMUP_STEPS}")
print(f"Seed: {SEED}")
print()
print(f"Use Liger: {USE_LIGER}")
print(f"Max Tokens per GPU: {MAX_TOKENS_PER_GPU:,}")
print(f"Max Sequence Length: {MAX_SEQ_LEN:,}")
print()
print(f"Save Final Checkpoint: {SAVE_FINAL_CHECKPOINT}")
print(f"Checkpoint at Epoch: {CHECKPOINT_AT_EPOCH}")
print()
print(f"Distributed: {NUM_GPUS} GPUs × {NUM_NODES} nodes = {NUM_GPUS * NUM_NODES} total GPUs")
print(f"Node Rank: {NODE_RANK}")
print(f"RDZV ID: {RDZV_ID}")
print(f"RDZV Endpoint: {RDZV_ENDPOINT}")

## MLflow Experiment Tracking

Launch an MLflow server to track training metrics, then configure the tracking URI below. Training loss, learning rate, and other metrics will be logged automatically.

To start the MLflow server:
```bash
mlflow server --host 0.0.0.0 --port 5000 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlflow-artifacts
```

In [ ]:
import os

################################################################################
# 📊 MLflow Configuration                                                      #
################################################################################
# Pull the secure URL that OpenShift AI automatically injected into the pod
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENT_NAME = "osft-continual-learning"
MLFLOW_RUN_NAME = f"osft-{BASE_MODEL.split('/')[-1]}-{ACTIVE_SCENARIO}"

print("📊 MLflow Configuration")
print("=" * 50)
print(f"Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {MLFLOW_EXPERIMENT_NAME}")
print(f"Run Name: {MLFLOW_RUN_NAME}")
print()

## Training with OSFT

With our hyperparameters configured, now we launch a training job and sit back while it enhances our new model 😎🍿

In [ ]:
print(
    """Please note that the cell will take a while to run (15-20 minutes on 2xL40). 
In the meantime, you can check the full training logs in the 
checkpoints-logs-dir/training_log_node0.log file and see per-step
metrics and progress in checkpoints-logs-dir/training_metrics_0.jsonl.
\n"""
)

print("🚀 Starting OSFT Continual Learning Training")
print("=" * 60)
print(f"Starting from: {BASE_MODEL}")
print(f"Training data: {DATASET_PATH}")
print(f"Output directory: {CHECKPOINTS_PATH}")
print(f"Unfreeze ratio: {UNFREEZE_RANK_RATIO}")
print()

# Capture output to prevent notebook crashes
output_buffer = StringIO()
error_buffer = StringIO()

training_start_time = time.time()

try:
    with redirect_stdout(output_buffer), redirect_stderr(error_buffer):
        # OSFT training
        training_result = osft(
            # Model and data
            model_path=BASE_MODEL,
            data_path=DATASET_PATH,
            ckpt_output_dir=CHECKPOINTS_PATH,
            
            # OSFT-specific
            unfreeze_rank_ratio=UNFREEZE_RANK_RATIO,
            
            # Training parameters
            num_epochs=NUM_EPOCHS,
            effective_batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            max_seq_len=MAX_SEQ_LEN,
            max_tokens_per_gpu=MAX_TOKENS_PER_GPU,
            
            # Data processing
            data_output_dir=DATA_OUTPUT_PATH,
            warmup_steps=WARMUP_STEPS,
            
            # Optimization
            use_liger=USE_LIGER,
            seed=SEED,
            lr_scheduler=LR_SCHEDULER,
            
            # Checkpointing
            checkpoint_at_epoch=CHECKPOINT_AT_EPOCH,
            save_final_checkpoint=SAVE_FINAL_CHECKPOINT,
            
            # Distributed training
            nproc_per_node=NUM_GPUS,
            nnodes=NUM_NODES,
            node_rank=NODE_RANK,
            rdzv_id=RDZV_ID,
            rdzv_endpoint=RDZV_ENDPOINT,

            # MLflow tracking
            mlflow_tracking_uri=MLFLOW_TRACKING_URI,
            mlflow_experiment_name=MLFLOW_EXPERIMENT_NAME,
            mlflow_run_name=MLFLOW_RUN_NAME,
        )
    
    training_duration = time.time() - training_start_time

    # Find the most recent checkpoint from Phase10 training
    final_checkpoint = find_most_recent_checkpoint(CHECKPOINTS_PATH)
    print(f"📁 Final model checkpoint: {final_checkpoint}")

    
    print(f"✅ OSFT training completed successfully in {training_duration/3600:.2f} hours!")
    print(f"📁 Checkpoint saved to: {final_checkpoint}")
    print()
    print("📊 Training Achievements:")
    print("  • Base model capabilities: ✅ Preserved")
    print("  • New knowledge integrated: ✅ Complete")
    print("  • Continual learning: ✅ Success")
    
except Exception as e:
    print(f"❌ OSFT training failed: {e}")
    print("\nError details:")
    print(error_buffer.getvalue())
    raise

### 🔬 Step 5: Model Evaluation (Base vs. Fine-Tuned)
Did the training actually work? To prove it, we will test the models using a "hold-out" set of questions that the model has **never seen before** (zero data leakage).

We will ask the same domain-specific questions to:
1. The original **Base Model** (Generic knowledge)
2. Our new **Fine-Tuned Model** (Specialized behavior)

We will compare answers to the ground truth answer from the datatable.

Watch how the fine-tuned model adopts the precise formatting, tone, and analytical rigor of our chosen industry dataset!

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

NUM_EVAL_SAMPLES = 3 
# 👇 Dynamically pulls the safe start index we set in Cell 4
EVAL_START_INDEX = CONF["eval_start"] 

def run_inference(model, tokenizer, user_text, conf, max_new_tokens=512):
    """Formats the prompt dynamically using the scenario config."""
    messages = [
        {"role": "system", "content": conf["system_prompt"]},
        {"role": "user", "content": f"{conf['user_prefix']}{user_text}"},
    ]
    
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.1, top_p=0.95)
    
    full_decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if "assistant" in full_decoded:
        return full_decoded.split("assistant")[-1].strip()
    return full_decoded

# --- Load Held-Out Data ---
print(f"📂 Fetching held-out samples from {CONF['dataset_name']}...")
eval_stream = load_dataset(CONF['dataset_name'], split=CONF['split'], streaming=True)

held_out_samples = []
# Skip the exact number of rows we trained on to guarantee zero data leakage
for row in eval_stream.skip(EVAL_START_INDEX).take(NUM_EVAL_SAMPLES):
    held_out_samples.append({
        "input_text": row[CONF["input_col"]],
        "ground_truth": row[CONF["target_col"]]
    })

# --- Evaluate Fine-Tuned Model ---
# (final_checkpoint is defined at the end of Cell 6)
print(f"\n🔄 Loading Fine-Tuned Model...")
ft_tokenizer = AutoTokenizer.from_pretrained(final_checkpoint)
ft_model = AutoModelForCausalLM.from_pretrained(final_checkpoint, torch_dtype=torch.bfloat16, device_map="cuda:0")

ft_answers = [run_inference(ft_model, ft_tokenizer, s["input_text"], CONF) for s in held_out_samples]

# 👇 Critical for your 1x L40S setup: Free up the VRAM before loading the base model
del ft_model
torch.cuda.empty_cache()

# --- Evaluate Base Model ---
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"\n🔄 Loading Base Model: {BASE_MODEL_NAME}...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda:0")

base_answers = [run_inference(base_model, base_tokenizer, s["input_text"], CONF) for s in held_out_samples]

# --- Display Results ---
print("\n" + "=" * 100)
print(f"🔬 EVALUATION RESULTS: {ACTIVE_SCENARIO.replace('_', ' ').upper()}")
print("=" * 100)

for i, sample in enumerate(held_out_samples):
    print(f"\n📄 SAMPLE #{i+1}")
    # Truncate input if it's a long article, show full if it's a short question
    display_input = sample['input_text'][:300] + "..." if len(sample['input_text']) > 300 else sample['input_text']
    print(f"INPUT: {CONF['user_prefix']}{display_input}")
    
    # 👇 Prints the full ground truth without slicing
    print(f"\n📌 GROUND TRUTH:\n{sample['ground_truth']}")
    
    print(f"\n🤖 FINE-TUNED MODEL:\n{ft_answers[i]}")
    print(f"\n📦 BASE MODEL:\n{base_answers[i]}")
    print("-" * 100)

## Evaluating Knowledge Retention with Standard Benchmarks

Instead of ad-hoc science questions, let's properly measure knowledge retention using established LLM benchmarks. We'll evaluate both the base model and our OSFT-trained model on:

- **MMLU** (Massive Multitask Language Understanding) — measures broad knowledge across 57 subjects
- **IFEval** (Instruction Following Evaluation) — measures instruction-following capability


This gives us a rigorous, reproducible comparison showing whether OSFT preserved the base model's capabilities.

In [ ]:
import os
import json
import glob
import subprocess
import math
import mlflow

# 1. Fetch Injected MLflow URI
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

mlflow.set_experiment("osft-continual-learning")

# 2. Configuration (20 samples per task for rapid execution)

# -------LIMITED TO 20 SAMPLES--------
BENCHMARKS = ["mmlu", "ifeval"]
EVAL_LIMIT = 20

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TRAINED_MODEL = "checkpoints-logs-dir/hf_format/samples_15000.0"

print("📊 Starting Fast Local Benchmark Evaluations (20 samples/task)\n" + "="*60)

def run_local_eval(model_path_or_id, benchmarks, label, limit=20):
    tasks = ",".join(benchmarks)
    output_dir = f"eval_results_{label}"
    
    # Explicitly force tokenizer=BASE_MODEL to prevent tokenizer format mismatches
    model_args = f"pretrained={model_path_or_id},tokenizer={BASE_MODEL},dtype=bfloat16,trust_remote_code=True"
    
    cmd = [
        "python", "-m", "lm_eval",
        "--model", "hf",
        "--model_args", model_args,
        "--tasks", tasks,
        "--apply_chat_template",
        "--output_path", output_dir,
    ]
    
    if limit:
        cmd.extend(["--limit", str(limit)])
        
    env = os.environ.copy()
    
    print(f"🔍 Fast Evaluating {label} [{model_path_or_id}] (limit={limit})...")
    print(f"   Executing command: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"⚠️  Evaluation failed (exit code {result.returncode}).")
        print(f"   stderr: {result.stderr[-1000:]}")
        return {}
        
    matches = sorted(glob.glob(f"{output_dir}/**/results*.json", recursive=True))
    if not matches:
        return {}
        
    results = {}
    with open(matches[-1]) as f:
        data = json.load(f)
        
    for task_name, task_results in data.get("results", {}).items():
        for acc_key in ("acc,none", "acc_norm,none", "exact_match,none", "prompt_level_strict_acc,none"):
            if acc_key in task_results:
                results[task_name] = task_results[acc_key]
                break
                
    return results

# 3. Evaluate Base Model & Log to MLflow
with mlflow.start_run(run_name=f"eval-base-{BASE_MODEL.split('/')[-1]}") as run_a:
    base_results = run_local_eval(BASE_MODEL, BENCHMARKS, label="base", limit=EVAL_LIMIT)
    if base_results:
        clean_metrics = {k.replace(",", "_").replace("/", "_"): v for k, v in base_results.items()}
        mlflow.log_metrics(clean_metrics)
        print("✅ Base model metrics successfully logged to MLflow!")

# 4. Evaluate OSFT Trained Model & Log to MLflow
with mlflow.start_run(run_name="eval-osft-trained-model") as run_b:
    trained_results = run_local_eval(TRAINED_MODEL, BENCHMARKS, label="trained", limit=EVAL_LIMIT)
    if trained_results:
        clean_metrics = {k.replace(",", "_").replace("/", "_"): v for k, v in trained_results.items()}
        mlflow.log_metrics(clean_metrics)
        print("✅ OSFT Trained model metrics successfully logged to MLflow!")

# 5. Formatted Retention Table
print("\n📊 Knowledge Retention Results (Sample size: 20 per task)")
print("=" * 70)
print(f"{'Benchmark':<40} {'Base Model':>12} {'OSFT Model':>14} {'Δ':>8}")
print("-" * 70)

all_benchmarks = sorted(set(list(base_results.keys()) + list(trained_results.keys())))

for benchmark in all_benchmarks:
    base_score = base_results.get(benchmark, float('nan'))
    trained_score = trained_results.get(benchmark, float('nan'))
    
    if not math.isnan(base_score) and not math.isnan(trained_score):
        delta = trained_score - base_score
        delta_str = f"{delta:+.1%}"
    else:
        delta_str = "N/A"
        
    b_str = f"{base_score:.1%}" if not math.isnan(base_score) else "N/A"
    t_str = f"{trained_score:.1%}" if not math.isnan(trained_score) else "N/A"
    
    print(f"{benchmark:<40} {b_str:>12} {t_str:>14} {delta_str:>8}")

print("\n🚀 Benchmarking complete! Both Base and OSFT runs are logged in MLflow.")

In [ ]:
import os
import json
import glob
import subprocess
import math
import mlflow

# 1. Fetch Injected MLflow URI
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

mlflow.set_experiment("osft-continual-learning")

# 2. Configuration (20 samples per task for rapid execution)

# -------------NO LIMIT FULL RUN---------------

BENCHMARKS = ["mmlu", "ifeval"]
EVAL_LIMIT = None

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TRAINED_MODEL = "checkpoints-logs-dir/hf_format/samples_15000.0"

print("📊 Starting Fast Local Benchmark Evaluations (20 samples/task)\n" + "="*60)

def run_local_eval(model_path_or_id, benchmarks, label, limit=None):
    tasks = ",".join(benchmarks)
    output_dir = f"eval_results_{label}"
    
    # Explicitly force tokenizer=BASE_MODEL to prevent tokenizer format mismatches
    model_args = f"pretrained={model_path_or_id},tokenizer={BASE_MODEL},dtype=bfloat16,trust_remote_code=True"
    
    cmd = [
        "python", "-m", "lm_eval",
        "--model", "hf",
        "--model_args", model_args,
        "--tasks", tasks,
        "--apply_chat_template",
        "--output_path", output_dir,
    ]
    
    if limit:
        cmd.extend(["--limit", str(limit)])
        
    env = os.environ.copy()
    
    print(f"🔍 Fast Evaluating {label} [{model_path_or_id}] (limit={limit})...")
    print(f"   Executing command: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"⚠️  Evaluation failed (exit code {result.returncode}).")
        print(f"   stderr: {result.stderr[-1000:]}")
        return {}
        
    matches = sorted(glob.glob(f"{output_dir}/**/results*.json", recursive=True))
    if not matches:
        return {}
        
    results = {}
    with open(matches[-1]) as f:
        data = json.load(f)
        
    for task_name, task_results in data.get("results", {}).items():
        for acc_key in ("acc,none", "acc_norm,none", "exact_match,none", "prompt_level_strict_acc,none"):
            if acc_key in task_results:
                results[task_name] = task_results[acc_key]
                break
                
    return results

# 3. Evaluate Base Model & Log to MLflow
with mlflow.start_run(run_name=f"eval-base-{BASE_MODEL.split('/')[-1]}") as run_a:
    base_results = run_local_eval(BASE_MODEL, BENCHMARKS, label="base", limit=EVAL_LIMIT)
    if base_results:
        clean_metrics = {k.replace(",", "_").replace("/", "_"): v for k, v in base_results.items()}
        mlflow.log_metrics(clean_metrics)
        print("✅ Base model metrics successfully logged to MLflow!")

# 4. Evaluate OSFT Trained Model & Log to MLflow
with mlflow.start_run(run_name="eval-osft-trained-model") as run_b:
    trained_results = run_local_eval(TRAINED_MODEL, BENCHMARKS, label="trained", limit=EVAL_LIMIT)
    if trained_results:
        clean_metrics = {k.replace(",", "_").replace("/", "_"): v for k, v in trained_results.items()}
        mlflow.log_metrics(clean_metrics)
        print("✅ OSFT Trained model metrics successfully logged to MLflow!")

# 5. Formatted Retention Table
print("\n📊 Knowledge Retention Results (Sample size: 20 per task)")
print("=" * 70)
print(f"{'Benchmark':<40} {'Base Model':>12} {'OSFT Model':>14} {'Δ':>8}")
print("-" * 70)

all_benchmarks = sorted(set(list(base_results.keys()) + list(trained_results.keys())))

for benchmark in all_benchmarks:
    base_score = base_results.get(benchmark, float('nan'))
    trained_score = trained_results.get(benchmark, float('nan'))
    
    if not math.isnan(base_score) and not math.isnan(trained_score):
        delta = trained_score - base_score
        delta_str = f"{delta:+.1%}"
    else:
        delta_str = "N/A"
        
    b_str = f"{base_score:.1%}" if not math.isnan(base_score) else "N/A"
    t_str = f"{trained_score:.1%}" if not math.isnan(trained_score) else "N/A"
    
    print(f"{benchmark:<40} {b_str:>12} {t_str:>14} {delta_str:>8}")

print("\n🚀 Benchmarking complete! Both Base and OSFT runs are logged in MLflow.")

## Final Analysis and Summary

In this notebook, we demonstrated how **Orthogonal Subspace Fine-Tuning (OSFT)** solves the continual learning challenge:

1. **New Capability Acquired**: The model learned strict JSON output formatting for table operations (TableGPT dataset)
2. **Knowledge Retained**: Standard benchmarks (MMLU, IFEval, GPQA Diamond) confirm the base model's capabilities are preserved
3. **Training Tracked**: MLflow captured all training metrics for reproducibility and comparison

OSFT enables models to learn new tasks while preserving their original knowledge through mathematical orthogonality — no catastrophic forgetting, verified by rigorous benchmark evaluation.